# Phase 1 Task B — Ablation 3.1: 90° Rotation — Swin-Tiny — Clean Dataset

**Experiment:** Discrete 90° rotation augmentation (0°, 90°, 180°, 270°) on duplicate-cleaned Task B
**Model:** Swin-Tiny
**Seed:** 42
**Batch:** 32
**Data:** `data/task-B-clean/`
**Output:** wandb `woodbench-taskB`, output dir `data/experiments/taskB_90rotation_clean/swin_tiny/`

So sánh với baseline M1 (random ±30° rotation) và bản 90° rotation gốc để đánh giá hiệu quả sau khi loại exact duplicate leakage.


In [1]:
# ── Cell 1: Root Detection ────────────────────────────────────
from pathlib import Path
import sys, os

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    NOTEBOOK = Path("__file__").resolve()
    ROOT = NOTEBOOK
    for _ in range(10):
        ROOT = ROOT.parent
        if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break

sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

[ROOT] /home/sakana/Code/ManifoldsTeam/woodbench


In [2]:
# ── Cell 2: Environment Setup ─────────────────────────────────
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
# for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops", "python-dotenv"]:
#     try: __import__(pkg.replace("-", "_"))
#     except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
os.environ.setdefault("HF_TOKEN", os.getenv("HF_TOKEN", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")
print(f"[HF] token: {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")

[.env] Loaded from /home/sakana/Code/ManifoldsTeam/woodbench/.env
[wandb] API key: set
[HF] token: set


In [3]:
# ── Cell 3: Clean Data Root ───────────────────────────────────
DATA_ROOT = ROOT / "data" / "task-B-clean"

if not (DATA_ROOT / "train").exists():
    raise FileNotFoundError(
        f"[ERROR] train/ not found in {DATA_ROOT}. "
        "Create the clean dataset from data/task-B before running this notebook."
    )

print(f"[Data] Found clean dataset at {DATA_ROOT}")

# Verify
train_dir = DATA_ROOT / "train"
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"[Data] {len(classes)} classes: {classes[:3]}...")
print(f"[Data] Train samples: {sum(1 for _ in train_dir.rglob('*.jpg')) + sum(1 for _ in train_dir.rglob('*.png'))}")


[Data] Found clean dataset at /home/sakana/Code/ManifoldsTeam/woodbench/data/task-B-clean
[Data] 20 classes: ['Araucaria araucana', 'Callitris columellaris', 'Calocedrus rupestris']...
[Data] Train samples: 3795


In [5]:
# ── Cell 4: DataLoaders (use_90rotation=True) ─────────────────
from src.data import get_dataloaders
tl, vl, testl = get_dataloaders(
    str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB",
    use_90rotation=True,   # ← ABLATION: discrete 0/90/180/270
)
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")
print(f"[Data] Augmentation: use_90rotation=True (discrete 0°/90°/180°/270°)")

[Data] Train=3795, Val=926, Test=851
[Data] Classes=20
[Data] Augmentation: use_90rotation=True (discrete 0°/90°/180°/270°)


In [6]:
# ── Cell 5: Imports & Device ──────────────────────────────────
import wandb, torch, numpy as np, json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function, WandbTracker
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

[Device] cuda


In [7]:
# ── Cell 6: train_one() — with WandbTracker + Test Evaluation ─

def train_one(backbone, seed, data_root, batch_size=32, epochs=50):
    """Train one model with 90° rotation augmentation.
    
    Improvements over M1 notebooks:
    1. WandbTracker passed to engine.fit() → epoch-level metrics logged
    2. Test evaluation after training → test_metrics.json saved
    3. use_90rotation=True in dataloaders
    """
    set_seed(seed)
    # Output to taskB_90rotation_clean/ (separate from original/leaky runs)
    output_dir = ROOT / "data" / "experiments" / "taskB_90rotation_clean" / backbone
    output_dir.mkdir(parents=True, exist_ok=True)

    # DataLoaders with 90° rotation
    train_loader, val_loader, test_loader = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB",
        use_90rotation=True,
    )
    num_classes = len(train_loader.dataset.classes)
    class_names = train_loader.dataset.classes
    print(f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}")

    # WandbTracker (logs epoch-level metrics automatically)
    config = {
        "backbone": backbone, "seed": seed, "epochs": epochs,
        "batch_size": batch_size, "num_classes": num_classes,
        "lr_head": 1e-4, "lr_backbone": 1e-5,
        "label_smoothing": 0.1, "scheduler": "cosine", "amp": True,
        "experiment": "taskB_90rotation_clean",
        "augmentation": "discrete_90 (0/90/180/270)",
        "dataset": "task-B-clean",
    }
    tracker = WandbTracker.start(
        project="woodbench-taskB", name=f"{backbone}_90rot_clean_s{seed}",
        config=config, dir_path=str(output_dir),
        tags=["ablation", "90rotation", "clean-data", backbone, f"seed{seed}"],
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True,
    )
    result = engine.fit(
        epochs=epochs, output_dir=output_dir,
        name=f"{backbone}_seed{seed}", save_best_only=True,
        early_stopping_patience=10, tracker=tracker,
    )

    # ── Test Evaluation ──────────────────────────────────────────
    print(f"\n[{backbone} s{seed}] Evaluating on test set...")
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            preds = model(images).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec  = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_per = f1_score(all_labels, all_preds, average=None, zero_division=0)

    print(f"[{backbone} s{seed}] TEST Acc={acc*100:.2f}%  F1={f1*100:.2f}%  Prec={prec*100:.2f}%  Rec={rec*100:.2f}%")

    # Log test metrics to wandb
    if tracker:
        tracker.log({
            "test/accuracy": acc, "test/f1_macro": f1,
            "test/precision_macro": prec, "test/recall_macro": rec,
            **{f"test/f1/{cn}": float(fi) for cn, fi in zip(class_names, f1_per)},
        }, step=result["epochs_trained"])

    # Save test_metrics.json
    test_metrics = {
        "backbone": backbone, "seed": seed,
        "experiment": "taskB_90rotation_clean",
        "best_val_acc": result["best_val_acc"],
        "best_epoch": result["best_epoch"],
        "epochs_trained": result["epochs_trained"],
        "stopped_early": result["stopped_early"],
        "test_accuracy": float(acc),
        "test_f1_macro": float(f1),
        "test_precision_macro": float(prec),
        "test_recall_macro": float(rec),
        "test_f1_per_class": {cn: float(fi) for cn, fi in zip(class_names, f1_per)},
    }
    with open(output_dir / "test_metrics.json", "w") as fp:
        json.dump(test_metrics, fp, indent=2)
    print(f"  → Saved: {output_dir / 'test_metrics.json'}")

    if tracker:
        tracker.finish()

    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return {**result, "test_metrics": test_metrics}

In [8]:
# ── Cell 7: Run Training (seed 42 only) ──────────────────────
SEED = 42

print(f"\n{'='*60}")
print(f"Ablation 3.1: 90° Rotation — Swin-Tiny | seed={SEED}")
print(f"{'='*60}")

result = train_one("swin_tiny", SEED, str(DATA_ROOT), batch_size=32, epochs=50)


Ablation 3.1: 90° Rotation — Swin-Tiny | seed=42
[swin_tiny s42] train=3795, classes=20


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ny0g507h3p (ny0g507h3p_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/50 | Train Loss: 3.1111, Acc: 2.41% | Val Loss: 3.1146, Acc: 1.30% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 1.30%)


Epoch 2/50 | Train Loss: 1.7159, Acc: 65.68% | Val Loss: 0.7560, Acc: 97.73% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 97.73%)


Epoch 3/50 | Train Loss: 0.7511, Acc: 97.75% | Val Loss: 0.6539, Acc: 99.68% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 99.68%)


Epoch 4/50 | Train Loss: 0.6617, Acc: 99.44% | Val Loss: 0.6356, Acc: 99.57% | LR: 8.00e-05[EarlyStop 1/10]


... (15 outputs left out)

Epoch 20/50 | Train Loss: 0.5963, Acc: 100.00% | Val Loss: 0.5966, Acc: 99.89% | LR: 7.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 20.

[swin_tiny s42] Evaluating on test set...


[swin_tiny s42] TEST Acc=99.76%  F1=99.76%  Prec=99.73%  Rec=99.80%
  → Saved: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/taskB_90rotation_clean/swin_tiny/test_metrics.json


lr,▁▃▅▆███████▇▇▇▇▇▇▆▆▆
test/accuracy,▁
test/f1/Araucaria araucana,▁
test/f1/Callitris columellaris,▁
test/f1/Calocedrus rupestris,▁
test/f1/Calocedrus sp,▁
test/f1/Chamaecyparis obtusa,▁
test/f1/Cryptomeria japonica,▁
test/f1/Cunninghamia lanceolata,▁
test/f1/Cupressus funebris (Cupressus pendula),▁
+19,...


[swin_tiny s42] val_acc=100.00%


In [9]:
# ── Cell 8: Summary ───────────────────────────────────────────
print("\n" + "="*60)
print("Ablation 3.1: 90° Rotation — Swin-Tiny — Clean Dataset — Summary")
print("="*60)
print(f"  Seed:           {SEED}")
print(f"  Val Acc (best):  {result['best_val_acc']:.2f}%")
print(f"  Best Epoch:      {result['best_epoch']}")
print(f"  Epochs Trained:  {result['epochs_trained']}")
print(f"  Early Stopped:   {result['stopped_early']}")
print(f"  ---")
tm = result['test_metrics']
print(f"  Test Accuracy:   {tm['test_accuracy']*100:.2f}%")
print(f"  Test F1 (macro): {tm['test_f1_macro']*100:.2f}%")
print(f"  Test Precision:  {tm['test_precision_macro']*100:.2f}%")
print(f"  Test Recall:     {tm['test_recall_macro']*100:.2f}%")
print(f"\n  Output: data/experiments/taskB_90rotation_clean/swin_tiny/")


Ablation 3.1: 90° Rotation — Swin-Tiny — Clean Dataset — Summary
  Seed:           42
  Val Acc (best):  100.00%
  Best Epoch:      10
  Epochs Trained:  20
  Early Stopped:   True
  ---
  Test Accuracy:   99.76%
  Test F1 (macro): 99.76%
  Test Precision:  99.73%
  Test Recall:     99.80%

  Output: data/experiments/taskB_90rotation_clean/swin_tiny/
